### General Imports

In [1]:
import pandas as pd
from pathlib import Path
import gc

### Data Processing functions

In [2]:
import os

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
print(REPO_ROOT)


/home/martinezyamamotoarthur/modeling_fraud_system


In [3]:
from src.data_processing.preprocessing import (
    run_preprocessing,
    run_processing_pipeline,
    split_train_validation_test,
)
from src.run_time_configuration import BaseConfigParams

## Load DF

In [4]:
SAMPLE_SIZE = 100000

In [5]:
if SAMPLE_SIZE:
    IEEE_DIR = Path("data/raw/ieee-fraud-detection")
    train_tx = pd.read_csv(IEEE_DIR / "train_transaction.csv")
    id_tx = pd.read_csv(IEEE_DIR / "train_identity.csv").assign(_has_identity=True)
    combined_df = train_tx.join(id_tx.set_index("TransactionID"), on="TransactionID", how="left").sample(SAMPLE_SIZE)
    del train_tx, id_tx
    gc.collect()


In [6]:
combined_df=run_preprocessing(combined_df)

In [9]:
display(combined_df)

,transaction_id,is_fraud,seconds_from_reference,amount_usd,product_channel,card_id,card2,card_issue_country,card_network,card5,...,id_36,id_37,id_38,device_type,device_info,_has_identity,transaction_datetime,weekdays,hours,flag_is_weekend
242529,3229529,0,5761033,30.95,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-05 16:17:13,0,16,False
271694,3258694,0,6583321,107.95,W,12501,490.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-02-15 04:42:01,3,4,False
360850,3347850,0,8957715,82.95,W,12567,555.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-03-14 16:15:15,2,16,False
381482,3368482,0,9560247,47.00,W,7680,360.0,150.0,mastercard,229.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-03-21 15:37:27,2,15,False
164522,3151522,0,3510438,57.95,W,17217,111.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-01-10 15:07:18,2,15,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282646,3269646,0,6900051,100.00,H,2039,562.0,150.0,mastercard,219.0,...,F,T,T,desktop,Windows,True,2018-02-18 20:40:51,6,20,True
451464,3438464,0,11529112,57.95,W,11207,361.0,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2018-04-13 10:31:52,4,10,False
126885,3113885,1,2509735,59.70,C,9633,130.0,185.0,visa,138.0,...,F,T,T,mobile,Moto G (5) Plus Build/NPNS25.137-15-11,True,2017-12-30 01:08:55,5,1,True
537318,3524318,0,14158061,125.00,R,1554,555.0,150.0,visa,226.0,...,F,T,F,mobile,SM-G930V Build/NRD90M,True,2018-05-13 20:47:41,6,20,True


In [ ]:
#split train, validation and test
runtime_config = BaseConfigParams(
    experiment_name="ieee_fraud",
    mlflow_run_name="ieee_data_transformation",
    training_start_date="2017-12-01",
    training_end_date="2018-03-31",
    test_start_date="2018-04-01",
    test_end_date="2018-06-30",
)
df_train, df_validation, df_test = split_train_validation_test(
    combined_df,
    runtime_config,
)
print(
    runtime_config.training_start_date,
    runtime_config.training_end_date,
    runtime_config.test_start_date,
    runtime_config.test_end_date,
)
print(len(df_train), len(df_validation), len(df_test))

In [ ]:
#run processing pipeline in the train+validation set
pipeline, df_train, df_validation, _ = run_processing_pipeline(df_train, df_validation)
print(df_train.shape, df_validation.shape)

In [ ]:
#fit pipeline in the test set
df_test = pipeline.transform(df_test)
print(df_test.shape)

In [ ]:
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
for name, frame in {
    "train": df_train,
    "validation": df_validation,
    "test": df_test,
}.items():
    path = PROCESSED_DIR / f"{name}.parquet"
    frame.to_parquet(path, index=False)
    print(f"wrote {path.resolve()} {frame.shape}")